In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import os

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
image_paths = []
mask_paths = []

image_folder = os.path.join(path, "dataset", "images")
mask_folder = os.path.join(path, "dataset", "masks")

for image in os.listdir(image_folder):
  image_paths.append(os.path.join(image_folder, image))

for mask in os.listdir(mask_folder):
  mask_paths.append(os.path.join(mask_folder, mask))


print(f"Total images: {len(image_paths)}")
print(f"Total masks: {len(mask_paths)}")

In [ ]:
image_paths[0]

In [ ]:
class UnderWaterDataset(Dataset):
  def __init__(self, image_paths, mask_paths, transform=None, target_transform=None):
    self.image_paths = image_paths
    self.mask_paths = mask_paths
    self.transform = transform
    self.target_transform = target_transform

  def __len__(self):
    # TODO: Return the number of samples in the dataset
    # YOUR CODE HERE
    return len(self.image_paths)

  def __getitem__(self, idx):
    # TODO: Load the image and mask at index idx
    # Hint: Use Image.open() and convert image to "RGB", mask to "L" (grayscale)

    # YOUR CODE HERE
    image = Image.open(self.image_paths[idx]).convert("RGB")
    mask = Image.open(self.mask_paths[idx]).convert("L")

    # Apply transforms
    if self.transform:
      image = self.transform(image)

    if self.target_transform:
      mask = self.target_transform(mask)
      mask = remap_mask(mask)  # Convert to binary mask

    return image, mask

In [ ]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

# Image transforms (Resize, ToTensor, Normalize with ImageNet stats)
image_transforms = transforms.Compose([
  transforms.ToTensor(),
  transforms.Resize((256, 256)),
])

# Mask transforms (Resize, PILToTensor)
mask_transforms = transforms.Compose([
  transforms.Resize((256, 256)),
  transforms.PILToTensor(),
])

In [ ]:
# Split into train and test sets (80% train, 20% test)

# YOUR CODE HERE
train_images, test_images, train_masks, test_masks = train_test_split(
  image_paths, mask_paths, test_size=0.2, random_state=42
)

# Create Dataset objects
train_dataset = UnderWaterDataset(train_images, train_masks, transform=image_transforms, target_transform=mask_transforms)
test_dataset = UnderWaterDataset(test_images, test_masks, transform=image_transforms, target_transform=mask_transforms)

# Create DataLoaders
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

In [ ]:
# Display 4 images with their masks side by side
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(4):

    image, mask = train_dataset[i]

    # Display image (denormalize first)
    axes[0, i].imshow(image.permute(1, 2, 0).numpy())
    axes[0, i].set_title(f"Image {i+1}")
    axes[0, i].axis("off")

    # Display mask
    axes[1, i].imshow(mask.squeeze(), cmap="gray")
    axes[1, i].set_title(f"Mask {i+1}")
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
!pip install -q segmentation_models_pytorch

In [ ]:
import segmentation_models_pytorch as smp

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
model = smp.Unet(
  encoder_name="efficientnet-b1",
  encoder_weights="imagenet",
  in_channels=3,
  classes=1,
).to(device)

model = model.to(device)

In [ ]:
import torch.optim as optim
from tqdm import tqdm

def train_one_epoch(model, dataloader, criterion, optimizer, device):
  model.train()
  total_loss = 0

  for images, masks in tqdm(dataloader):
    # Move data to device
    images, masks = images.to(device), masks.to(device).float()

    # TODO: Complete the training step
    # 1. Forward pass
    # 2. Compute loss
    # 3. Zero gradients
    # 4. Backward pass
    # 5. Update weights

    # YOUR CODE HERE
    outputs = model(images)
    loss = criterion(outputs, masks)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item()

  return total_loss / len(dataloader)

In [ ]:
# TODO: Task 6 (continued) - Complete the validation function

def validate(model, dataloader, criterion, device):
  model.eval()
  total_loss = 0

  with torch.no_grad():
    for images, masks in dataloader:
      images, masks = images.to(device), masks.to(device).float()

      # TODO: Complete the validation step
      # 1. Forward pass
      # 2. Compute loss

      # YOUR CODE HERE
      outputs = model(images)
      loss = criterion(outputs, masks)

      total_loss += loss.item()

  return total_loss / len(dataloader)

In [ ]:
from torch import nn

# YOUR CODE HERE
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

num_epochs = 2  # Train for 5 epochs

In [ ]:
# Run training
train_losses = []
val_losses = []

for epoch in range(num_epochs):
  train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
  val_loss = validate(model, test_loader, criterion, device)

  train_losses.append(train_loss)
  val_losses.append(val_loss)

  print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

In [ ]:
# TODO: Task 8 - Visualize predictions on test images
import random

model.eval()

fig, axes = plt.subplots(4, 3, figsize=(12, 16))

# Get random test samples
indices = random.sample(range(len(test_dataset)), 4)

for i, idx in enumerate(indices):
  image, mask = test_dataset[idx]

  # TODO: Get model prediction
  # 1. Add batch dimension: image.unsqueeze(0)
  # 2. Move to device
  # 3. Get prediction: model(image)
  # 4. Apply sigmoid to get probabilities
  # 5. Threshold at 0.5 to get binary mask

  with torch.no_grad():
    input_tensor = image.unsqueeze(0).to(device)
    output = model(input_tensor)
    pred = torch.sigmoid(output)
    pred = (pred > 0.5).float().cpu()

  # Display results
  axes[i, 0].imshow(image.permute(1, 2, 0).numpy())
  axes[i, 0].set_title("MRI Image")
  axes[i, 0].axis("off")

  axes[i, 1].imshow(mask.squeeze(), cmap="gray")
  axes[i, 1].set_title("Ground Truth")
  axes[i, 1].axis("off")

  axes[i, 2].imshow(pred.squeeze(), cmap="gray")
  axes[i, 2].set_title("Prediction")
  axes[i, 2].axis("off")

plt.tight_layout()
plt.show()